In [ ]:
# Convert Feather to Parquet
def convert_feather_to_parquet_simple(feather_dir, parquet_dir):
    """
    Convert all .feather files in 'feather_dir' to .parquet in 'parquet_dir'
    while skipping hidden or '._' files. Uses pandas for reading/writing.
    """
    os.makedirs(parquet_dir, exist_ok=True)
    
    # Gather all .feather files and filter out hidden ones
    feather_files = glob.glob(os.path.join(feather_dir, "*.feather"))
    # Skip files that start with '._' or '.' to avoid hidden files
    feather_files = [
        f for f in feather_files
        if not os.path.basename(f).startswith("._")
        and not os.path.basename(f).startswith(".")
    ]
    feather_files.sort()  # Sort lexicographically
    
    total_files = len(feather_files)
    print(f"Found {total_files} Feather files in {feather_dir}.")
    if total_files == 0:
        return
    
    # Show first and last for sanity check
    print("First file:", os.path.basename(feather_files[0]))
    print("Last file: ", os.path.basename(feather_files[-1]))

    start_time = time.time()
    successes = 0
    failures = 0
    failed_files = []

    for i, file_path in enumerate(feather_files, start=1):
        base_name = os.path.basename(file_path)
        print(f"[{i}/{total_files}] Converting: {base_name}", end="")

        try:
            # Read Feather with pandas
            df = pd.read_feather(file_path)
            
            # Construct the output file path
            parquet_name = base_name.replace(".feather", ".parquet")
            out_path = os.path.join(parquet_dir, parquet_name)
            
            # Write Parquet (you can specify compression if you like, e.g. compression='snappy')
            df.to_parquet(out_path, index=False)
            
            successes += 1
            elapsed = time.time() - start_time
            print(f" ✓  (elapsed: {elapsed:.1f}s)")
            
        except Exception as e:
            failures += 1
            failed_files.append((file_path, str(e)))
            print(f" ✗  Error: {e}")
    
    elapsed = time.time() - start_time
    print(f"\nDone! {successes} succeeded, {failures} failed.")
    print(f"Total time: {elapsed:.1f}s, avg {elapsed/total_files:.2f}s per file.")

    if failed_files:
        print("\nFailed files:")
        for path, err in failed_files:
            print(f" - {os.path.basename(path)}: {err}")

if __name__ == "__main__":
    feather_input = "/Volumes/T7/bid-volume-data-feather-sorted"
    parquet_output = "/Volumes/T7/bid-volume-data-parquet-sorted"

    convert_feather_to_parquet_simple(feather_input, parquet_output)